> Versión de portafolio. Se conservan las salidas históricas del trabajo académico. Las rutas absolutas de datos, cuando existían, se sustituyeron por nombres locales. Esta versión no se ha ejecutado de nuevo; consulte README y DATOS.md antes de reproducir.


# Módulo 52 · Pruebas de hipótesis en regresión múltiple
**Javier Bueno Sánchez · Profesión Científico de Datos**

**Pregunta:** ¿qué atributos ayudan a pronosticar el precio de las viviendas de King County? Se comparan selección hacia adelante, eliminación hacia atrás y selección bidireccional, con pruebas *t* calculadas mediante matrices. Los precios y resultados se expresan en miles de dólares estadounidenses (USD).

## 1. Diseño, variables y cautelas
El archivo tiene 21,613 observaciones. Se separa una muestra de prueba del 20% con semilla fija (42) antes de seleccionar predictores. La prueba **no participa** en la selección. `id` es un identificador, `date` se excluye porque no se plantea un modelo temporal, y `zipcode` es una categoría codificada numéricamente cuya inclusión lineal implicaría un orden artificial. `sqft_above` y `sqft_basement` se excluyen porque `sqft_living = sqft_above + sqft_basement`, una dependencia lineal exacta. `yr_renovated` se transforma en indicador de renovación y `yr_built` en edad al cierre aproximado de 2015. `lat` y `long` son coordenadas geográficas, usadas como aproximación lineal a localización. `price` se divide entre 1000; las superficies del lote se dividen entre 1000 para facilitar la lectura.

Se usa α = 0.05 para **entrada** y salida en los tres métodos. La prueba bivariada de entrada evalúa el coeficiente condicional al conjunto ya incluido. Cada procedimiento es determinista; si hubiera empate, se usa el orden alfabético. Las pruebas *p* posteriores a selección son exploratorias y no tienen la cobertura confirmatoria habitual debido a la selección repetida. Tampoco muestran causalidad.

In [2]:
from pathlib import Path
if not Path('kc_house_data.csv').exists():
    options=[Path('upload/kc_house_data.csv'),Path('../../upload/kc_house_data.csv')]
    found=next((p for p in options if p.exists()),None)
    if found is None: raise FileNotFoundError('Coloca kc_house_data.csv junto al notebook antes de ejecutarlo.')
    import shutil
    shutil.copyfile(found,'kc_house_data.csv')

In [3]:
import numpy as np, pandas as pd
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import matplotlib.pyplot as plt
pd.set_option('display.max_rows', 40)
df = pd.read_csv('kc_house_data.csv')
print('Dimensión:',df.shape, 'Nulos:',int(df.isna().sum().sum()), 'Duplicados id:',int(df.id.duplicated().sum()))
print('Identidad exacta de superficies:',bool((df.sqft_living==df.sqft_above+df.sqft_basement).all()))
print(df[['price','bedrooms','bathrooms','sqft_living']].describe().round(1))

Dimensión: (21613, 21) Nulos: 0 Duplicados id: 177
Identidad exacta de superficies: True
           price  bedrooms  bathrooms  sqft_living
count    21613.0   21613.0    21613.0      21613.0
mean    540088.1       3.4        2.1       2079.9
std     367127.2       0.9        0.8        918.4
min      75000.0       0.0        0.0        290.0
25%     321950.0       3.0        1.8       1427.0
50%     450000.0       3.0        2.2       1910.0
75%     645000.0       4.0        2.5       2550.0
max    7700000.0      33.0        8.0      13540.0


In [4]:
base=['bedrooms','bathrooms','sqft_living','sqft_lot','floors','waterfront','view','condition','grade','lat','long','sqft_living15','sqft_lot15']
X=df[base].copy().astype(float)
X['age_2015']=2015-df.yr_built
X['renovated']= (df.yr_renovated>0).astype(float)
X[['sqft_lot','sqft_lot15']]/=1000
y=df.price.astype(float)/1000
assert np.isfinite(X.to_numpy()).all() and np.isfinite(y).all()
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.20,random_state=42)
names=list(X.columns); n=len(Xtr)
# Solo productos cruzados: las pruebas usan X'X, X'y e y'y.
D=np.column_stack([np.ones(n),Xtr.to_numpy()]); yy=ytr.to_numpy()
G=D.T@D; gy=D.T@yy; y2=yy@yy
print('Entrenamiento / prueba:',len(Xtr),len(Xte),'· predictores candidatos:',len(names))

Entrenamiento / prueba: 17290 4323 · predictores candidatos: 15


## 2. Estadístico de prueba matricial
Con una columna de unos en $X$, para cada conjunto de variables $S$ se calcula $\hat\beta=(X_S^{\mathsf T}X_S)^{-1}X_S^{\mathsf T}y$ mediante resolución del sistema lineal, $SSE=y^{\mathsf T}y-\hat\beta^{\mathsf T}X_S^{\mathsf T}y$, $s^2=SSE/(n-p)$ y $\mathrm{Var}(\hat\beta)=s^2(X_S^{\mathsf T}X_S)^{-1}$. Para cada variable $j$, $H_0:\beta_j=0$ frente a $H_1:\beta_j\ne0$, $t_j=\hat\beta_j/SE(\hat\beta_j)$ y $p_j=2P(T_{n-p}\ge |t_j|)$. Aquí $p$ incluye el intercepto. Usamos `solve`, sin formar explícitamente la inversa para estimar $\beta$.

In [6]:
def fit_matrix(selected):
    idx=[0]+[names.index(c)+1 for c in selected]
    gram=G[np.ix_(idx,idx)]; cross=gy[idx]
    beta=np.linalg.solve(gram,cross)
    sse=max(float(y2-beta@cross),0)
    dof=n-len(idx)
    cov=(sse/dof)*np.linalg.solve(gram,np.eye(len(idx)))
    se=np.sqrt(np.diag(cov))
    t=beta/se
    pv=2*stats.t.sf(np.abs(t),dof)
    return dict(beta=beta,se=se,t=t,pv=pv,sse=sse,dof=dof)
def candidate_p(selected,new):
    f=fit_matrix(selected+[new]); return float(f['pv'][-1])
def forward(alpha=.05):
    selected=[]; trace=[]
    while True:
        remaining=sorted(set(names)-set(selected))
        if not remaining: break
        c,p=min(((c,candidate_p(selected,c)) for c in remaining),key=lambda a:(a[1],a[0]))
        if p>=alpha: break
        selected.append(c); trace.append(('entra',c,p))
    return selected,trace
def backward(alpha=.05):
    selected=names.copy();trace=[]
    while selected:
        f=fit_matrix(selected)
        c,p=max(zip(selected,f['pv'][1:]),key=lambda a:(a[1],a[0]))
        if p<=alpha:break
        selected.remove(c);trace.append(('sale',c,float(p)))
    return selected,trace
def bidirectional(alpha=.05):
    selected=[];trace=[]
    while True:
        remaining=sorted(set(names)-set(selected))
        if not remaining:break
        c,p=min(((c,candidate_p(selected,c)) for c in remaining),key=lambda a:(a[1],a[0]))
        if p>=alpha:break
        selected.append(c);trace.append(('entra',c,p))
        # Tras cada entrada se revisan todas las variables existentes.
        while selected:
            f=fit_matrix(selected)
            worst,pmax=max(zip(selected,f['pv'][1:]),key=lambda a:(a[1],a[0]))
            if pmax<=alpha:break
            selected.remove(worst);trace.append(('sale',worst,float(pmax)))
    return selected,trace
results={ 'Hacia adelante':forward(), 'Hacia atrás':backward(), 'Bidireccional':bidirectional() }
for method,(selected,trace) in results.items():
    print('\n',method, '·',len(selected),'variables: ', ', '.join(selected))
    print(pd.DataFrame(trace,columns=['decisión','variable','p']).to_string(index=False,float_format=lambda z:f'{z:.3g}'))


 Hacia adelante · 14 variables:  bathrooms, age_2015, grade, sqft_living, lat, waterfront, view, bedrooms, condition, sqft_living15, long, renovated, sqft_lot15, floors
decisión      variable         p
   entra     bathrooms         0
   entra      age_2015         0
   entra         grade         0
   entra   sqft_living         0
   entra           lat         0
   entra    waterfront         0
   entra          view 4.54e-121
   entra      bedrooms  2.06e-52
   entra     condition  5.29e-19
   entra sqft_living15  1.71e-08
   entra          long  6.48e-13
   entra     renovated  6.68e-08
   entra    sqft_lot15  4.92e-05
   entra        floors  0.000227

 Hacia atrás · 14 variables:  bedrooms, bathrooms, sqft_living, floors, waterfront, view, condition, grade, lat, long, sqft_living15, sqft_lot15, age_2015, renovated
decisión variable     p
    sale sqft_lot 0.107

 Bidireccional · 14 variables:  bathrooms, age_2015, grade, sqft_living, lat, waterfront, view, bedrooms, condition, sq

**Lectura verbal:** hacia adelante comienza con solo el intercepto y agrega el predictor con menor valor *p* condicional mientras sea inferior a 0.05. Hacia atrás comienza con todos los candidatos y elimina el mayor valor *p* superior a 0.05. El método bidireccional incorpora como el primero y, tras cada incorporación, revisa si debe retirar predictores que perdieron significancia. Si una variable sale, eso solo indica falta de evidencia adicional dadas las demás variables del modelo y este umbral.

In [8]:
def score(selected):
    f=fit_matrix(selected)
    pred=f['beta'][0]+Xte[selected].to_numpy()@f['beta'][1:]
    r2train=1-f['sse']/np.sum((yy-yy.mean())**2)
    adj=1-(1-r2train)*(n-1)/(n-len(selected)-1)
    return {'predictores':len(selected),'R2 entreno':r2train,'R2 ajustado':adj,'R2 prueba':r2_score(yte,pred),'MAE prueba (USD)':mean_absolute_error(yte,pred)*1000,'RMSE prueba (USD)':np.sqrt(mean_squared_error(yte,pred))*1000}
comparison=pd.DataFrame({m:score(s) for m,(s,t) in results.items()}).T
print(comparison.round(3).to_string())
chosen=min(results,key=lambda m:comparison.loc[m,'RMSE prueba (USD)'])
selected=results[chosen][0]
print('Modelo documentado:',chosen)
f=fit_matrix(selected)
ci=np.column_stack((f['beta']-stats.t.ppf(.975,f['dof'])*f['se'],f['beta']+stats.t.ppf(.975,f['dof'])*f['se']))
coef=pd.DataFrame({'variable':['intercepto']+selected,'coef_miles_USD':f['beta'],'error_estandar':f['se'],'t':f['t'],'p':f['pv'],'IC95_inf':ci[:,0],'IC95_sup':ci[:,1]})
print(coef.to_string(index=False,float_format=lambda z:f'{z:.5g}'))

                predictores  R2 entreno  R2 ajustado  R2 prueba  MAE prueba (USD)  RMSE prueba (USD)
Hacia adelante         14.0       0.694        0.694      0.695        128199.638         214855.128
Hacia atrás            14.0       0.694        0.694      0.695        128199.638         214855.128
Bidireccional          14.0       0.694        0.694      0.695        128199.638         214855.128
Modelo documentado: Hacia adelante
     variable  coef_miles_USD  error_estandar       t           p  IC95_inf  IC95_sup
   intercepto          -36723          1649.6 -22.263 2.8198e-108    -39957    -33490
    bathrooms          41.762          3.5914  11.628  3.8711e-31    34.722    48.801
     age_2015          2.5481        0.080182  31.779 1.9089e-215     2.391    2.7053
        grade          98.111           2.365  41.485           0    93.476    102.75
  sqft_living         0.16677       0.0037389  44.604           0   0.15944    0.1741
          lat          545.85          11.566

## 3. Comprobación automatizada y multicolinealidad
Contrastamos el ajuste matricial y sus valores *p* con `statsmodels.OLS` sobre la misma muestra y variables. Calculamos VIF para los predictores seleccionados (VIF alto indica inestabilidad de coeficientes). El modelo puede predecir razonablemente aunque los coeficientes individuales sean inestables. Los intervalos de confianza son nominales y no corrigen el proceso de selección.

In [10]:
auto=sm.OLS(ytr,sm.add_constant(Xtr[selected],has_constant='add')).fit()
assert np.allclose(auto.params.to_numpy(),f['beta'],rtol=1e-6,atol=1e-5)
assert np.allclose(auto.pvalues.to_numpy(),f['pv'],rtol=1e-4,atol=1e-7)
print('Máxima diferencia absoluta de coeficientes:',np.max(np.abs(auto.params.to_numpy()-f['beta'])))
print('R2 ajustado automatizado:',auto.rsquared_adj)
vifs=pd.Series({c:variance_inflation_factor(Xtr[selected].to_numpy(),i) for i,c in enumerate(selected)}).sort_values(ascending=False)
# Para VIF se agrega intercepto, imprescindible con latitud/longitud y otros datos no centrados.
vif_design=sm.add_constant(Xtr[selected],has_constant='add').to_numpy()
vifs=pd.Series({c:variance_inflation_factor(vif_design,i+1) for i,c in enumerate(selected)}).sort_values(ascending=False)
print('VIF con intercepto:\n',vifs.round(2).to_string())
print('Máximo VIF:',round(vifs.max(),2))

Máxima diferencia absoluta de coeficientes: 2.2005231585353613e-05
R2 ajustado automatizado: 0.6940037788399487
VIF con intercepto:
 sqft_living      4.98
grade            3.31
bathrooms        3.28
sqft_living15    2.87
age_2015         2.41
bedrooms         1.65
floors           1.62
long             1.45
view             1.37
condition        1.23
waterfront       1.19
renovated        1.15
sqft_lot15       1.11
lat              1.11
Máximo VIF: 4.98


## 4. Error y alcance del resultado
Las métricas de prueba son una evaluación predictiva en observaciones reservadas. La significancia estadística no implica que una variable mejore mucho el pronóstico ni que su efecto sea causal. Las coordenadas lineales no recogen todos los patrones espaciales; precios extremos, heterocedasticidad y posibles relaciones no lineales limitan el modelo. Una partición aleatoria puede compartir viviendas o zonas relacionadas entre entrenamiento y prueba; para una aplicación real conviene evaluar una separación temporal o espacial. La comparación de tres métodos en la misma prueba también introduce una pequeña selección sobre la prueba: para validar el método elegido haría falta otra muestra independiente.

**Interpretación de los signos:** cada coeficiente es un cambio promedio condicionado a los otros predictores, medido en miles de USD por unidad de la variable; no debe interpretarse un coeficiente aislado de `bedrooms` o `sqft_living` como valoración comercial sin revisar su colinealidad y plausibilidad.

In [12]:
pred=f['beta'][0]+Xte[selected].to_numpy()@f['beta'][1:]
resid=yte.to_numpy()-pred
fig,axs=plt.subplots(1,2,figsize=(11,4.2))
axs[0].scatter(pred,yte,s=5,alpha=.24,color='#3A7BD5'); lim=[min(pred.min(),yte.min()),max(pred.max(),yte.max())];axs[0].plot(lim,lim,color='#263238',lw=1)
axs[0].set(xlabel='Precio pronosticado (miles USD)',ylabel='Precio real (miles USD)',title='Prueba: predicción frente a realidad')
axs[1].scatter(pred,resid,s=5,alpha=.24,color='#3A7BD5');axs[1].axhline(0,color='#263238',lw=1)
axs[1].set(xlabel='Precio pronosticado (miles USD)',ylabel='Real menos pronosticado',title='Prueba: residuales')
fig.tight_layout();fig.savefig('diagnostico_modulo52.png',dpi=150,bbox_inches='tight');plt.show()
print('Interpretación: RMSE penaliza más los errores grandes que MAE; revisar la dispersión y los valores extremos antes de usar pronósticos individuales.')

Interpretación: RMSE penaliza más los errores grandes que MAE; revisar la dispersión y los valores extremos antes de usar pronósticos individuales.


### Ecuación y ejemplo reproducibles

In [14]:
terms=[f'({b:.4f} × {name})' for name,b in zip(selected,f['beta'][1:])]
print('Precio estimado (miles USD) =',f'{f["beta"][0]:.4f}', '+ ' + ' + '.join(terms))
example=Xte.iloc[[0]][selected];calc=float(f['beta'][0]+example.to_numpy()@f['beta'][1:])
print('Ejemplo primera fila de prueba: pronóstico USD',round(calc*1000,2),'· real USD',round(float(yte.iloc[0]*1000),2))
print('Fuente: kc_house_data.csv, conjunto de viviendas King County publicado en Kaggle; cálculo propio.')

Precio estimado (miles USD) = -36723.4898 + (41.7617 × bathrooms) + (2.5481 × age_2015) + (98.1115 × grade) + (0.1668 × sqft_living) + (545.8478 × lat) + (572.5892 × waterfront) + (47.9282 × view) + (-33.0799 × bedrooms) + (27.2647 × condition) + (0.0300 × sqft_living15) + (-81.2404 × long) + (42.8027 × renovated) + (-0.2266 × sqft_lot15) + (13.1645 × floors)
Ejemplo primera fila de prueba: pronóstico USD 466398.45 · real USD 365000.0
Fuente: kc_house_data.csv, conjunto de viviendas King County publicado en Kaggle; cálculo propio.
